In [1]:
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
import json
from pathlib import Path

rosemary = pd.read_csv("../data/cultivar_compound_catalogue.csv").drop_duplicates("compound_id")
curated = pd.read_csv("../data/curated_chembl.csv")
splits = json.loads(Path("../splits/split_indices.json").read_text())

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

# Get training actives
train_actives = curated.iloc[splits["indices"]["train"]]
train_actives = train_actives[train_actives["activity_class"] == "active"]
train_fps = [morgan_gen.GetFingerprint(Chem.MolFromSmiles(s)) for s in train_actives["canonical_smiles"]]

# Calculate similarity
sim_results = []
for _, row in rosemary.iterrows():
    mol = Chem.MolFromSmiles(row["canonical_smiles"])
    fp = morgan_gen.GetFingerprint(mol)
    sims = DataStructs.BulkTanimotoSimilarity(fp, train_fps)
    max_sim = max(sims) if sims else 0.0
    sim_support = max(0, min(1, max_sim))
    
    sim_results.append({
        "compound_id": row["compound_id"],
        "max_similarity": max_sim,
        "similarity_support": sim_support
    })

pd.DataFrame(sim_results).to_csv("../results/similarity_scores.csv", index=False)
print("Similarity scores saved.")

Similarity scores saved.
